# Adversarial Search: Minimax Algorithm for Tic‑Tac‑Toe

**Author:** Phabel Antonio López Delgado, BSc.

This notebook implements a fully playable Tic‑Tac‑Toe game with an AI opponent powered by the **Minimax algorithm with alpha‑beta pruning**. The AI is unbeatable, playing optimally in every situation.

We cover:

- **Game mechanics** – board representation, move validation, win/draw detection.
- **Minimax algorithm** – recursive search that assumes both players play optimally.
- **Alpha‑beta pruning** – optimisation that reduces the number of nodes evaluated, making the search efficient.
- **Heuristic evaluation** – simple scoring based on completed lines.

**Key Techniques & Libraries:**
- `math` – for infinity values in minimax.
- No external libraries are required; the implementation uses only Python built‑ins.

**Objective:**
To understand adversarial search and the minimax algorithm, and to apply it to a simple turn‑based game. This serves as a foundation for more complex game‑playing AI (e.g., chess, Go).

In [1]:
import math

print("All imports successful.")

All imports successful.


## Tic‑Tac‑Toe Game Class

We define a `TicTacToe` class that manages the board state, move validation, win detection, and the minimax search. The AI plays as 'O' and the human plays as 'X'.

In [2]:
class TicTacToe:
    """
    A Tic‑Tac‑Toe game with an AI using minimax with alpha‑beta pruning.
    The AI is 'O', the human is 'X'.
    """
    def __init__(self):
        self.board = [["_", "_", "_"],
                      ["_", "_", "_"],
                      ["_", "_", "_"]]

    def print_board(self):
        """Print the current board in a readable format."""
        for row in self.board:
            print(" | ".join(row))
            print("-" * 9)

    def move(self, row, col, player):
        """
        Place a player's mark on the board.
        Returns True if successful, False if the cell is occupied.
        """
        if self.board[row][col] == "_":
            self.board[row][col] = player
            return True
        return False

    def check_winner(self):
        """
        Check if there is a winner.
        Returns the winning player ('X' or 'O') or None if no winner yet.
        """
        # Check rows
        for i in range(3):
            if self.board[i][0] == self.board[i][1] == self.board[i][2] != "_":
                return self.board[i][0]
        # Check columns
        for i in range(3):
            if self.board[0][i] == self.board[1][i] == self.board[2][i] != "_":
                return self.board[0][i]
        # Check diagonals
        if self.board[0][0] == self.board[1][1] == self.board[2][2] != "_":
            return self.board[0][0]
        if self.board[0][2] == self.board[1][1] == self.board[2][0] != "_":
            return self.board[0][2]
        return None

    def evaluate_board(self):
        """
        Heuristic evaluation of the current board.
        Positive score favours 'O' (AI), negative favours 'X' (human).
        Sums scores for each complete line (row, column, diagonal).
        """
        score = 0
        # Check rows
        for i in range(3):
            if self.board[i][0] == self.board[i][1] == self.board[i][2] == "O":
                score += 10
            elif self.board[i][0] == self.board[i][1] == self.board[i][2] == "X":
                score -= 10
        # Check columns
        for i in range(3):
            if self.board[0][i] == self.board[1][i] == self.board[2][i] == "O":
                score += 10
            elif self.board[0][i] == self.board[1][i] == self.board[2][i] == "X":
                score -= 10
        # Check main diagonal
        if self.board[0][0] == self.board[1][1] == self.board[2][2] == "O":
            score += 10
        elif self.board[0][0] == self.board[1][1] == self.board[2][2] == "X":
            score -= 10
        # Check anti-diagonal
        if self.board[0][2] == self.board[1][1] == self.board[2][0] == "O":
            score += 10
        elif self.board[0][2] == self.board[1][1] == self.board[2][0] == "X":
            score -= 10
        return score

    def is_board_full(self):
        """Return True if there are no empty cells left."""
        for row in self.board:
            if "_" in row:
                return False
        return True

    def minimax(self, depth, is_maximizing, alpha, beta):
        """
        Minimax algorithm with alpha‑beta pruning.
        Returns the evaluation score for the current board state.
        """
        # Terminal states: leaf nodes or win/draw
        if depth == 0:
            return self.evaluate_board()

        winner = self.check_winner()
        if winner is not None or self.is_board_full():
            if winner == "O":
                return 10 + depth   # AI wins – prefer shorter wins
            elif winner == "X":
                return -10 - depth  # Human wins – prefer longer losses
            else:
                return 0            # Draw

        # Maximising player (AI: 'O')
        if is_maximizing:
            max_eval = -math.inf
            for i in range(3):
                for j in range(3):
                    if self.board[i][j] == "_":
                        self.board[i][j] = "O"
                        eval_score = self.minimax(depth - 1, False, alpha, beta)
                        self.board[i][j] = "_"
                        max_eval = max(max_eval, eval_score)
                        alpha = max(alpha, eval_score)
                        if beta <= alpha:
                            break
            return max_eval

        # Minimising player (Human: 'X')
        else:
            min_eval = math.inf
            for i in range(3):
                for j in range(3):
                    if self.board[i][j] == "_":
                        self.board[i][j] = "X"
                        eval_score = self.minimax(depth - 1, True, alpha, beta)
                        self.board[i][j] = "_"
                        min_eval = min(min_eval, eval_score)
                        beta = min(beta, eval_score)
                        if beta <= alpha:
                            break
            return min_eval

    def best_move(self):
        """
        Determine the best move for the AI ('O') using minimax.
        Returns a tuple (row, column) of the optimal move.
        """
        best_score = -math.inf
        best_move = None
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == "_":
                    self.board[i][j] = "O"
                    move_score = self.minimax(depth=9, is_maximizing=False,
                                              alpha=-math.inf, beta=math.inf)
                    self.board[i][j] = "_"
                    if move_score > best_score:
                        best_score = move_score
                        best_move = (i, j)
        return best_move

    def play(self):
        """
        Main game loop. Human plays as 'X', AI as 'O'.
        """
        current_player = "X"  # Human starts first

        while True:
            self.print_board()

            if current_player == "X":
                # Human move
                while True:
                    try:
                        row = int(input("Player X, enter row (0, 1, or 2): "))
                        col = int(input("Player X, enter column (0, 1, or 2): "))
                        if self.move(row, col, current_player):
                            break
                        else:
                            print("Invalid move. Cell is already occupied.")
                    except ValueError:
                        print("Please enter integers (0, 1, or 2).")
            else:
                # AI move
                if self.is_board_full():
                    print("It's a draw!")
                    self.print_board()
                    break
                print("Computer's turn (O)...")
                row, col = self.best_move()
                self.move(row, col, "O")

            # Check for win or draw after the move
            winner = self.check_winner()
            if winner == "X":
                print("Player X wins!")
                self.print_board()
                break
            elif winner == "O":
                print("Computer wins!")
                self.print_board()
                break
            elif self.is_board_full():
                print("It's a draw!")
                self.print_board()
                break

            # Switch player
            current_player = "O" if current_player == "X" else "X"

In [3]:
game = TicTacToe()
game.play()

_ | _ | _
---------
_ | _ | _
---------
_ | _ | _
---------
Player X, enter row (0, 1, or 2): 0
Player X, enter column (0, 1, or 2): 0
X | _ | _
---------
_ | _ | _
---------
_ | _ | _
---------
Computer's turn (O)...
X | _ | _
---------
_ | O | _
---------
_ | _ | _
---------
Player X, enter row (0, 1, or 2): 2
Player X, enter column (0, 1, or 2): 2
X | _ | _
---------
_ | O | _
---------
_ | _ | X
---------
Computer's turn (O)...
X | O | _
---------
_ | O | _
---------
_ | _ | X
---------
Player X, enter row (0, 1, or 2): 2
Player X, enter column (0, 1, or 2): 1
X | O | _
---------
_ | O | _
---------
_ | X | X
---------
Computer's turn (O)...
X | O | _
---------
_ | O | _
---------
O | X | X
---------
Player X, enter row (0, 1, or 2): 0
Player X, enter column (0, 1, or 2): 2
X | O | X
---------
_ | O | _
---------
O | X | X
---------
Computer's turn (O)...
X | O | X
---------
_ | O | O
---------
O | X | X
---------
Player X, enter row (0, 1, or 2): 1
Player X, enter column (0, 1, or

## Summary and Next Steps

**Accomplished:**
- Implemented a fully playable **Tic‑Tac‑Toe** game with a human vs. computer mode.
- Designed an AI opponent using the **Minimax algorithm with alpha‑beta pruning** for efficient decision making.
- The AI evaluates the board using a custom heuristic score, exploring all possible moves up to depth 9 (full game).
- Alpha‑beta pruning significantly reduces the number of nodes evaluated, making the AI responsive even for a full game tree.
- The game handles win/draw detection, turn switching, and invalid move validation.

**Key Results:**
- The AI (playing as "O") consistently chooses optimal moves, never losing if it has a winning strategy.
- In the example game shown, the AI played optimally and the game ended in a **draw**, which is the expected outcome when both players play perfectly.
- The minimax algorithm with pruning evaluates the game tree efficiently enough for real‑time play.

**Key Insights:**
- **Minimax** is a fundamental adversarial search algorithm that assumes both players play optimally. It chooses moves that maximise the AI's chance of winning while minimising the opponent's.
- **Alpha‑beta pruning** removes branches that cannot influence the final decision, reducing the search space from O(b^d) to O(b^(d/2)) in the best case.
- The evaluation function (heuristic) is crucial: here we used a simple score based on completed lines, but more advanced heuristics could improve play in more complex games.
- The recursive nature of minimax makes it easy to implement but may hit recursion limits for deeper games; however, for Tic‑Tac‑Toe (depth 9) it is perfectly fine.
- The game implementation separates logic (board state, moves, win detection) from the user interface (console input/output), making it easy to adapt to a graphical interface.

**Suggested Next Steps:**
1. **Add a difficulty level** – vary the search depth or introduce randomness to make the AI beatable.
2. **Implement a GUI** – use `tkinter`, `pygame`, or a web interface for a more interactive experience.
3. **Extend to other games** – apply the same minimax framework to Connect Four, Checkers, or Othello.
4. **Improve the evaluation function** – consider center control, fork opportunities, and block opponent forks for stronger play.
5. **Implement iterative deepening** – to handle time constraints for deeper games.
6. **Add move ordering** – to improve alpha‑beta pruning efficiency (e.g., evaluate promising moves first).
7. **Create a tournament mode** – where the AI plays against itself to verify optimality.
8. **Visualise the search tree** – show the minimax decision process step by step.

**Reflection:**
This notebook provides a hands‑on implementation of adversarial search using the classic game of Tic‑Tac‑Toe. The minimax algorithm with alpha‑beta pruning is a cornerstone of AI for zero‑sum games, and this implementation illustrates its core principles clearly. The modular design allows easy extension to other board games, and the console interface is simple yet functional. The example game ending in a draw confirms that both players played optimally – a hallmark of a well‑implemented minimax AI. This exercise builds a strong foundation for understanding more complex game‑playing AI, such as those used in chess or Go.